# Access an ASAM ODS EXD-API Plugin

## Prepare Python Environment to Access GRPC Service

In [18]:
import os
import pathlib

from ods_exd_api_box.proto import ods_external_data_pb2, ods_external_data_pb2_grpc
import grpc
from google.protobuf.json_format import MessageToJson

## EXD-API

The EXD-API plugin is running as a RPC service at a given URL.
Running `external_data_file.py` will run the plugin at the given URL.

In [19]:
exd_api_plugin_url = "localhost:50051"

## Import Phase

We will open a MDF4 file using the EXD-API and extract the internal structure of the file to import it into the ASAM ODS server.

In [20]:
data_file_path = os.path.abspath("data/test5.hdf5")
if not os.path.exists(data_file_path):
    raise Exception("Data file is missing")

In [21]:
import_file_url = pathlib.Path(data_file_path).as_uri()
import_file_parameters = ""
print(import_file_url)

# Will be filled from Structure
access_file_url = None
access_file_parameters = None

file:///c:/Users/AKR/github/asam_ods_exd_api_microunidaq/data/test5.hdf5


### Extract Infos from Structure

The structure contains infos about groups and channels to create corresponding measurements, submatrices and measurement_quantities

In [22]:
with grpc.insecure_channel(exd_api_plugin_url) as channel:
    stub = ods_external_data_pb2_grpc.ExternalDataReaderStub(channel)

    # import file into ASAM ODS Server physical storage
    import_identifier = ods_external_data_pb2.Identifier(url=import_file_url, parameters=import_file_parameters)

    import_handle = stub.Open(import_identifier)
    try:
        structure = stub.GetStructure(ods_external_data_pb2.StructureRequest(handle=import_handle))
        print(MessageToJson(structure))

        access_file_url = structure.identifier.url
        access_file_parameters = structure.identifier.parameters

        for group in structure.groups:
            group_id = group.id
            for channel in group.channels:
                channel_id = channel.id
    finally:
        stub.Close(import_handle)

{
  "identifier": {
    "url": "file:///c:/Users/AKR/github/asam_ods_exd_api_microunidaq/data/test5.hdf5"
  },
  "name": "test5.hdf5",
  "groups": [
    {
      "name": "SinewaveGenerator",
      "totalNumberOfChannels": "2",
      "numberOfRows": "114688",
      "channels": [
        {
          "name": "Time",
          "dataType": "DT_DOUBLE",
          "unitString": "s",
          "attributes": {
            "variables": {
              "independent": {
                "longArray": {
                  "values": [
                    1
                  ]
                }
              }
            }
          }
        },
        {
          "id": "1",
          "name": "SinewaveGenerator",
          "dataType": "DT_FLOAT",
          "unitString": "V"
        }
      ],
      "attributes": {
        "variables": {
          "Input": {
            "stringArray": {
              "values": [
                "AIN 1"
              ]
            }
          },
          "IEPE": {
     

## Access Bulk Data

With the stored information the ASAM ODS server can access the bulk data from the EXD-API plugin

In [23]:
with grpc.insecure_channel(exd_api_plugin_url) as channel:
    stub = ods_external_data_pb2_grpc.ExternalDataReaderStub(channel)

    # info from physical storage
    access_group_id = 0
    access_channel_ids = [0, 1]
    access_identifier = ods_external_data_pb2.Identifier(url=access_file_url, parameters=access_file_parameters)

    # open bulk access
    access_handle = stub.Open(access_identifier)
    try:
        request = ods_external_data_pb2.ValuesRequest(
            handle=access_handle, group_id=access_group_id, channel_ids=access_channel_ids
        )

        # read first chunk
        request.start = 0
        request.limit = 3
        values = stub.GetValues(request)
        print(MessageToJson(values))

        # read second chunk
        request.start = 3
        request.limit = 10
        values = stub.GetValues(request)
        print(MessageToJson(values))

    finally:
        stub.Close(access_handle)

{
  "channels": [
    {
      "values": {
        "dataType": "DT_DOUBLE",
        "doubleArray": {
          "values": [
            0.0,
            2e-05,
            4e-05
          ]
        }
      }
    },
    {
      "id": "1",
      "values": {
        "dataType": "DT_FLOAT",
        "floatArray": {
          "values": [
            -10.3756895,
            -10.749884,
            -10.955344
          ]
        }
      }
    }
  ]
}
{
  "channels": [
    {
      "values": {
        "dataType": "DT_DOUBLE",
        "doubleArray": {
          "values": [
            6.000000000000001e-05,
            8e-05,
            0.0001,
            0.00012000000000000002,
            0.00014000000000000001,
            0.00016,
            0.00018,
            0.0002,
            0.00022,
            0.00024000000000000003
          ]
        }
      }
    },
    {
      "id": "1",
      "values": {
        "dataType": "DT_FLOAT",
        "floatArray": {
          "values": [
            

## Plot All Groups

Load every group from the file and display each one in its own tab as an XY (Time vs Value) plot.

In [24]:
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display
from google.protobuf.json_format import MessageToDict

group_tabs = []
group_titles = []

with grpc.insecure_channel(exd_api_plugin_url) as channel:
    stub = ods_external_data_pb2_grpc.ExternalDataReaderStub(channel)

    # Fetch structure to enumerate all groups
    identifier = ods_external_data_pb2.Identifier(url=import_file_url, parameters=import_file_parameters)
    handle = stub.Open(identifier)
    try:
        structure = stub.GetStructure(ods_external_data_pb2.StructureRequest(handle=handle))
    finally:
        stub.Close(handle)

    # One gRPC connection per group to read bulk data
    for group in structure.groups:
        group_id = group.id
        channel_ids = [ch.id for ch in group.channels]

        # Derive axis labels from channel metadata
        x_channel = next((ch for ch in group.channels if ch.id == 0), None)
        y_channel = next((ch for ch in group.channels if ch.id == 1), None)
        x_label = f"{x_channel.name} [{x_channel.unit_string}]" if x_channel else "X"
        y_label = f"{y_channel.name} [{y_channel.unit_string}]" if y_channel else "Y"

        # Read all data for this group in one request (no limit = all rows)
        acc_handle = stub.Open(
            ods_external_data_pb2.Identifier(url=structure.identifier.url, parameters=structure.identifier.parameters)
        )
        try:
            req = ods_external_data_pb2.ValuesRequest(
                handle=acc_handle,
                group_id=group_id,
                channel_ids=channel_ids,
                start=0,
                limit=group.number_of_rows,
            )
            values = stub.GetValues(req)
        finally:
            stub.Close(acc_handle)

        # Extract X and Y arrays (channel 0 = time/double, channel 1 = data/float)
        channels_dict = {ch_val.id: ch_val for ch_val in values.channels}
        x_vals = list(channels_dict[0].values.double_array.values) if 0 in channels_dict else []
        y_vals = list(channels_dict[1].values.float_array.values) if 1 in channels_dict else []

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=x_vals, y=y_vals, mode="lines", name=group.name))
        fig.update_layout(
            title=group.name,
            xaxis_title=x_label,
            yaxis_title=y_label,
            margin=dict(l=60, r=20, t=40, b=40),
        )

        group_tabs.append(go.FigureWidget(fig))
        group_titles.append(group.name)

tab = widgets.Tab(children=group_tabs)
for i, title in enumerate(group_titles):
    tab.set_title(i, title)

display(tab)


    'data': [{'mode': 'lines',
              'name': 'SinewaveGenerator',
       …